# 01 — Data Loader & Returns

This notebook demonstrates Module 1:
- Download OHLCV data via `quantlab.data.loader`
- Compute daily/log/cumulative returns
- Print a performance summary table
- Compare against SPY benchmark

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from quantlab.data.loader import load_ohlcv
from quantlab.features.returns import (
    calculate_returns,
    calculate_cumulative_returns,
)
from quantlab.evaluation.metrics import performance_summary

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True

## 1. Download data

In [ ]:
SYMBOLS  = ['AAPL', 'MSFT', 'NVDA', 'SPY']
START    = '2020-01-01'
END      = '2025-01-01'
CACHE    = '../data/cache'

data = load_ohlcv(SYMBOLS, start_date=START, end_date=END, cache_dir=CACHE)
print(data.shape)
data.head(8)

## 2. Compute returns per symbol

In [ ]:
returns = (
    data['close']
    .unstack('symbol')
    .apply(calculate_returns)
)
returns.head()

## 3. Plot cumulative returns

In [ ]:
cum_returns = returns.apply(calculate_cumulative_returns)

fig, ax = plt.subplots()
for col in cum_returns.columns:
    lw = 2.5 if col == 'SPY' else 1.5
    ls = '--' if col == 'SPY' else '-'
    ax.plot(cum_returns.index, cum_returns[col] * 100, label=col, linewidth=lw, linestyle=ls)

ax.set_title('Cumulative Returns (%) — 2020–2025')
ax.set_ylabel('Return (%)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/cumulative_returns.png', dpi=150)
plt.show()

## 4. Performance summary table

In [ ]:
spy_returns = returns['SPY'].dropna()

rows = []
for symbol in [s for s in SYMBOLS if s != 'SPY']:
    r = returns[symbol].dropna()
    summary = performance_summary(r, benchmark_returns=spy_returns)
    summary['symbol'] = symbol
    rows.append(summary)

summary_df = pd.DataFrame(rows).set_index('symbol')
fmt = {
    'total_return': '{:.1%}',
    'annualized_return': '{:.1%}',
    'annualized_volatility': '{:.1%}',
    'sharpe_ratio': '{:.2f}',
    'max_drawdown': '{:.1%}',
    'alpha_annualized': '{:.1%}',
    'beta': '{:.2f}',
    'information_ratio': '{:.2f}',
}
summary_df.style.format(fmt)

## 5. Next steps

- `02_features_and_factors.ipynb` — build technical factors on this data
- `03_backtest_basic.ipynb` — simple moving-average strategy backtest